# Calculate Box-Whisker Data for Georgia

In [1]:
import geopandas as gpd
import pandas as pd
import pickle
import json

## Loading Plans

In [2]:
gdf = gpd.read_file('../inputs/ga_seawulf.gpkg')
with open('../outputs/ga_raceblind_5000.pkl', 'rb') as file:
    raceblind_pickle = pickle.load(file)
with open('../outputs/ga_vra_5000.pkl', 'rb') as file:
    vra_pickle = pickle.load(file)

In [3]:
node_to_id = gdf['UNIQUE_ID'].to_dict()

In [5]:
vra_plans = []
for plan in vra_pickle:
    df = pd.DataFrame(plan.items(), columns=['node_id', 'District'])
    df['UNIQUE_ID'] = df['node_id'].map(node_to_id)
    df = df[['UNIQUE_ID', 'District']]
    vra_plans.append(df)

In [4]:
raceblind_plans = []
for plan in raceblind_pickle:
    df = pd.DataFrame(plan.items(), columns=['node_id', 'District'])
    df['UNIQUE_ID'] = df['node_id'].map(node_to_id)
    df = df[['UNIQUE_ID', 'District']]
    raceblind_plans.append(df)

## Loading Enacted Plan Data

In [6]:
with open('../../preprocessing/output/Georgia/ga-enacted-shares.json') as json_file:
    enacted_plan = json.load(json_file)

enacted_plan

{'white': {'share': {'0': 0.19709280821604497,
   '1': 0.2458373413353966,
   '2': 0.2978006524330315,
   '3': 0.3102098055642323,
   '4': 0.39941997315513433,
   '5': 0.5212707283411053,
   '6': 0.5758916377067114,
   '7': 0.5791427929152464,
   '8': 0.6090559182366511,
   '9': 0.6248018328743742,
   '10': 0.6371708595976929,
   '11': 0.6436685765667803,
   '12': 0.6668470269952361,
   '13': 0.6998964889901926}},
 'black': {'share': {'0': 0.07747370732300228,
   '1': 0.11442163801159272,
   '2': 0.11864034629137826,
   '3': 0.11911362047220425,
   '4': 0.226108822483846,
   '5': 0.23315171008590618,
   '6': 0.27536898620769873,
   '7': 0.29724127475376927,
   '8': 0.3611880763681228,
   '9': 0.47544426684371555,
   '10': 0.4902703698814722,
   '11': 0.4962372702369252,
   '12': 0.49786639516844694,
   '13': 0.5018323539867422}},
 'latino': {'share': {'0': 0.05628411158277744,
   '1': 0.05946516767585413,
   '2': 0.06310642813826561,
   '3': 0.07168660212040735,
   '4': 0.0760621953976

In [23]:
enacted_plan_demo = {}
demographics = ['white', 'black', 'latino', 'other']

for demo in demographics:
    shares = enacted_plan[demo]["share"]

    df = pd.DataFrame({
        "enacted": shares
    })

    df["districtIndex"] = range(1, len(df) + 1)

    enacted_plan_demo[demo] = df

enacted_plan_demo['white']

,enacted,districtIndex
0,0.197093,1
1,0.245837,2
2,0.297801,3
3,0.310210,4
4,0.399420,5
5,0.521271,6
6,0.575892,7
7,0.579143,8
8,0.609056,9
9,0.624802,10


## Standard Box-Whisker

### VRA

In [25]:
vra_pop_shares = []

for i, plan in enumerate(vra_plans):
    gdf_plan = gdf.merge(plan, on="UNIQUE_ID")
    gdf_plan = gdf_plan.drop(columns=['District_x'])
    gdf_plan = gdf_plan.rename(columns={'District_y': 'District'})

    district_stats = (
        gdf_plan.groupby("District")
        .sum(numeric_only=True)
        .assign(
            white_share=lambda df: df['White_population'] / df['Total_population'],
            black_share=lambda df: df['Black_population'] / df['Total_population'],
            latino_share=lambda df: df['Latino_population'] / df['Total_population'],
            other_share=lambda df: df['Other_population'] / df['Total_population'],
            plan_id=i
        )
        .drop(columns=['plan_id', 'Kamala D. Harris', 'Donald J. Trump', 'Other_candidates', 
                       'Total_votes', 'White_population', 'Black_population', 'Latino_population', 
                       'Other_population', 'Total_population'])
        .reset_index()
    )

    vra_pop_shares.append(district_stats)

In [26]:
vra_pop_shares[200]

,District,white_share,black_share,latino_share,other_share
0,1,0.656252,0.266013,0.049445,0.028290
1,2,0.452464,0.472816,0.037986,0.036735
2,3,0.637687,0.152807,0.088966,0.120540
3,4,0.178717,0.694492,0.060276,0.066515
4,5,0.697012,0.179961,0.068509,0.054518
5,6,0.518982,0.332662,0.059681,0.088676
6,7,0.250792,0.628498,0.074624,0.046087
7,8,0.596511,0.306229,0.053012,0.044247
8,9,0.837412,0.055080,0.076168,0.031340
9,10,0.533974,0.382126,0.038821,0.045079


In [27]:
vra_summaries = {}

for demo in demographics:
    sorted_plans = []

    for df in vra_pop_shares:
        df_sorted = df.sort_values(f"{demo}_share").reset_index(drop=True)
        df_sorted.drop(columns='District')
        df_sorted['districtIndex'] = df_sorted.index + 1
        sorted_plans.append(df_sorted)

    all_plans_df = pd.concat(sorted_plans, ignore_index=True)
    vra_summaries[demo] = (
        all_plans_df.groupby("districtIndex")[f"{demo}_share"]
        .agg(
            min="min",
            q1=lambda x: x.quantile(0.25),
            median="median",
            q3=lambda x: x.quantile(0.75),
            max="max"
        )
        .reset_index()
    )

    vra_summaries[demo] = vra_summaries[demo].merge(enacted_plan_demo[demo], on="districtIndex", how="left")

vra_summaries['white']

,districtIndex,min,q1,median,q3,max,enacted
0,1,0.077727,0.209354,0.257952,0.291092,0.380264,0.197093
1,2,0.212395,0.317180,0.341830,0.370834,0.469836,0.245837
2,3,0.289813,0.381157,0.405524,0.429071,0.500404,0.297801
3,4,0.357738,0.433479,0.457437,0.484613,0.519321,0.310210
4,5,0.426543,0.479389,0.492285,0.501905,0.558914,0.399420
5,6,0.452464,0.500382,0.511310,0.528504,0.559073,0.521271
6,7,0.481195,0.528828,0.536810,0.548301,0.616466,0.575892
7,8,0.510008,0.548557,0.574302,0.585667,0.656282,0.579143
8,9,0.533802,0.586881,0.592096,0.609391,0.663489,0.609056
9,10,0.570420,0.627167,0.641459,0.654681,0.684360,0.624802


### RaceBlind

In [8]:
raceblind_pop_shares = []

for i, plan in enumerate(raceblind_plans):
    gdf_plan = gdf.merge(plan, on="UNIQUE_ID")
    gdf_plan = gdf_plan.drop(columns=['District_x'])
    gdf_plan = gdf_plan.rename(columns={'District_y': 'District'})

    district_stats = (
        gdf_plan.groupby("District")
        .sum(numeric_only=True)
        .assign(
            white_share=lambda df: df['White_population'] / df['Total_population'],
            black_share=lambda df: df['Black_population'] / df['Total_population'],
            latino_share=lambda df: df['Latino_population'] / df['Total_population'],
            other_share=lambda df: df['Other_population'] / df['Total_population'],
            plan_id=i
        )
        .drop(columns=['plan_id', 'Kamala D. Harris', 'Donald J. Trump', 'Other_candidates', 
                       'Total_votes', 'White_population', 'Black_population', 'Latino_population', 
                       'Other_population', 'Total_population'])
        .reset_index()
    )

    raceblind_pop_shares.append(district_stats)

In [9]:
raceblind_pop_shares[200]

,District,white_share,black_share,latino_share,other_share
0,1,0.617860,0.282579,0.055961,0.043600
1,2,0.422522,0.322497,0.100882,0.154098
2,3,0.675374,0.231061,0.046470,0.047095
3,4,0.821418,0.065272,0.081324,0.031987
4,5,0.231693,0.664112,0.048077,0.056117
5,6,0.493493,0.370994,0.081621,0.053892
6,7,0.688360,0.166357,0.071892,0.073390
7,8,0.548813,0.365058,0.042932,0.043197
8,9,0.284620,0.597922,0.054224,0.063235
9,10,0.553177,0.381423,0.039224,0.026177


In [28]:
raceblind_summaries = {}

for demo in demographics:
    sorted_plans = []

    for df in raceblind_pop_shares:
        df_sorted = df.sort_values(f"{demo}_share").reset_index(drop=True)
        df_sorted.drop(columns='District')
        df_sorted['districtIndex'] = df_sorted.index + 1
        sorted_plans.append(df_sorted)

    all_plans_df = pd.concat(sorted_plans, ignore_index=True)
    raceblind_summaries[demo] = (
        all_plans_df.groupby("districtIndex")[f"{demo}_share"]
        .agg(
            min="min",
            q1=lambda x: x.quantile(0.25),
            median="median",
            q3=lambda x: x.quantile(0.75),
            max="max"
        )
        .reset_index()
    )

    raceblind_summaries[demo] = raceblind_summaries[demo].merge(enacted_plan_demo[demo], on="districtIndex", how="left")

raceblind_summaries['white']

,districtIndex,min,q1,median,q3,max,enacted
0,1,0.105062,0.212144,0.240761,0.271531,0.392740,0.197093
1,2,0.211430,0.284794,0.315491,0.348611,0.448599,0.245837
2,3,0.273775,0.381989,0.415809,0.447096,0.524165,0.297801
3,4,0.357738,0.456018,0.477723,0.499183,0.549519,0.310210
4,5,0.411544,0.496995,0.512858,0.526924,0.556722,0.399420
5,6,0.456241,0.521730,0.533658,0.547137,0.563901,0.521271
6,7,0.496163,0.543087,0.549172,0.553868,0.616466,0.575892
7,8,0.517257,0.550411,0.555866,0.568079,0.630657,0.579143
8,9,0.539521,0.570761,0.596547,0.612718,0.662383,0.609056
9,10,0.551985,0.613365,0.623046,0.634631,0.686857,0.624802


### Merge to Json to Send to Backend

In [37]:
def create_json(vra_summaries, raceblind_summaries, demographics):
    data = {'state': 'Georgia'}

    vra = {}
    raceblind = {}

    for demo in demographics:
        vra[demo] = vra_summaries[demo].to_dict(orient="records")
        raceblind[demo] = raceblind_summaries[demo].to_dict(orient="records")

    data['vra'] = vra
    data['raceBlind'] = raceblind

    return data

In [38]:
data = create_json(vra_summaries, raceblind_summaries, demographics)

with open("./output/Georgia/box_whisker/ga-box-whisker.json", "w") as f:
    json.dump(data, f, indent=2)

## Minority Effectiveness Box-Whisker